In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")


from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# SMOTE fixes the class imbalance (more approvals than rejections in dataset)
from imblearn.over_sampling import SMOTE

# SHAP explains why the model made each decision
import shap

# For saving the trained model to a file
import joblib

In [ ]:
df = pd.read_csv("cleaned_dataset.csv")

FEATURES = [
    "gender", "married", "dependents", "education", "self_employed",
    "applicant_income", "co_applicant_income", "loan_amount",
    "loan_amount_term", "credit_history", "property_area"
]

TARGET = "loan_status"

X = df[FEATURES]

categorical_columns = [
    "gender", "married", "dependents", "education",
    "self_employed", "property_area"
]
X = pd.get_dummies(X, columns=categorical_columns, drop_first=True)

y = df[TARGET]

print("Dataset shape:", df.shape)
print("Loan Status counts:")
print(y.value_counts())
print(f"Approval rate: {y.mean():.1%}")

In [9]:
# Convert the loan status labels from strings to numeric values so mean() works
y = y.map({"N": 0, "Y": 1})

print("Dataset shape:", df.shape)
print("Loan Status counts:")
print(y.value_counts())
print(f"Approval rate: {y.mean():.1%}")

Dataset shape: (614, 12)
Loan Status counts:
loan_status
1    422
0    192
Name: count, dtype: int64
Approval rate: 68.7%


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows:     {len(X_test)}")

Training rows: 491
Test rows:     123


In [12]:
# Fix class imbalance with SMOTE

sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE: ", y_train_bal.value_counts().to_dict())

ValueError: could not convert string to float: 'Male'

In [41]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  LogisticRegression(max_iter=2000, random_state=42))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,  # how much each tree corrects errors
        subsample=0.8,
        random_state=42,
        eval_metric="logloss",
        verbosity=0
    ),


    "LightGBM": LGBMClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        num_leaves=31,
        random_state=42,
        verbosity=-1
    ),
}

trained_models = {}

print("Training models...")
for name, model in models.items():
    model.fit(X_train_bal, y_train_bal)
    trained_models[name] = model


print("\nAll models trained.")

Training models...

All models trained.


In [42]:
results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model":     name,
        "Accuracy":  round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall":    round(recall_score(y_test, y_pred), 4),
        "F1 Score":  round(f1_score(y_test, y_pred), 4),
        "ROC-AUC":   round(roc_auc_score(y_test, y_prob), 4),
    })

# Builds a comparison table and sort by ROC-AUC (best model at top)
df_results = pd.DataFrame(results).set_index("Model")
df_results = df_results.sort_values("ROC-AUC", ascending=False)

print(" Model comparison table")
print(df_results.to_string())

 Model comparison table
                     Accuracy  Precision  Recall  F1 Score  ROC-AUC
Model                                                              
LightGBM               0.8049     0.8861  0.8235    0.8537   0.8619
Logistic Regression    0.8293     0.8636  0.8941    0.8786   0.8588
Random Forest          0.8455     0.8667  0.9176    0.8914   0.8545
XGBoost                0.8049     0.8675  0.8471    0.8571   0.8198
Decision Tree          0.7642     0.8684  0.7765    0.8199   0.8121


In [43]:
best_name  = df_results.index[0]
best_model = trained_models[best_name]

print(f"\nBest model: {best_name}")
print("=" * 50)

y_pred_best = best_model.predict(X_test)

# Classification report — precision, recall, F1 per class
print(classification_report(
    y_test, y_pred_best,
    target_names=["Rejected", "Approved"]
))

# Confusion matrix — shows correct vs incorrect predictions
cm = confusion_matrix(y_test, y_pred_best)
print("Confusion Matrix:")
print(f"  Correct Rejections : {cm[0][0]}  |  Wrong Approvals : {cm[0][1]}")
print(f"  Missed Rejections  : {cm[1][0]}  |  Correct Approvals: {cm[1][1]}")


Best model: LightGBM
              precision    recall  f1-score   support

    Rejected       0.66      0.76      0.71        38
    Approved       0.89      0.82      0.85        85

    accuracy                           0.80       123
   macro avg       0.77      0.79      0.78       123
weighted avg       0.82      0.80      0.81       123

Confusion Matrix:
  Correct Rejections : 29  |  Wrong Approvals : 9
  Missed Rejections  : 15  |  Correct Approvals: 70


In [44]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    best_model, X, y,
    cv=cv,
    scoring="roc_auc"
)

print(f"\nCross-Validation — {best_name}")
print(f"ROC-AUC per fold: {[round(s, 4) for s in cv_scores]}")
print(f"Mean: {cv_scores.mean():.4f}  |  Std: {cv_scores.std():.4f}")


Cross-Validation — LightGBM
ROC-AUC per fold: [np.float64(0.8211), np.float64(0.7709), np.float64(0.7225), np.float64(0.7277), np.float64(0.7456)]
Mean: 0.7576  |  Std: 0.0360


In [45]:
# Saves the best model
joblib.dump(best_model, "best_model.joblib")
print(f"\nModel saved as: best_model.joblib")




Model saved as: best_model.joblib


In [46]:
# Risk scoring engine


def get_risk_category(risk_score):
    if risk_score <= 30:
        return "Low Risk"
    elif risk_score <= 60:
        return "Medium Risk"
    else:
        return "High Risk"


def get_recommendation(risk_score):
    if risk_score <= 30:
        return "Approve — strong candidate"
    elif risk_score <= 60:
        return "Review — officer should verify manually"
    else:
        return "Reject — high default probability"


# Score every applicant in the test set
y_prob_best = best_model.predict_proba(X_test)[:, 1]

risk_scores      = [(1 - p) * 100 for p in y_prob_best]
risk_categories  = [get_risk_category(s) for s in risk_scores]
recommendations  = [get_recommendation(s) for s in risk_scores]

# Builds a results table
df_risk = pd.DataFrame({
    "Approval Probability": y_prob_best.round(4),
    "Risk Score":           [round(s, 1) for s in risk_scores],
    "Risk Category":        risk_categories,
    "Recommendation":       recommendations,
    "Actual Outcome":       y_test.values,
})

df_risk["Actual Label"] = df_risk["Actual Outcome"].map({1: "Approved", 0: "Rejected"})

print("\nSample Risk Scores (first 8 applicants):")
print(df_risk.head(8).to_string(index=False))


Sample Risk Scores (first 8 applicants):
 Approval Probability  Risk Score Risk Category                          Recommendation  Actual Outcome Actual Label
               0.0001       100.0     High Risk       Reject — high default probability               0     Rejected
               0.8145        18.5      Low Risk              Approve — strong candidate               1     Approved
               0.3450        65.5     High Risk       Reject — high default probability               1     Approved
               0.4247        57.5   Medium Risk Review — officer should verify manually               1     Approved
               0.8924        10.8      Low Risk              Approve — strong candidate               1     Approved
               0.5294        47.1   Medium Risk Review — officer should verify manually               0     Rejected
               0.9859         1.4      Low Risk              Approve — strong candidate               1     Approved
               0.9681 

In [47]:
# Risk distribution summary:  Shows how many applicants fell into each risk category and whether the categories actually match real-world outcomes. If Low Risk applicants are approved at 90%+ in reality,the risk engine is working correctly.

summary = df_risk.groupby("Risk Category").agg(
    Count           = ("Risk Score", "count"),
    Avg_Risk_Score  = ("Risk Score", "mean"),
    Actual_Approval = ("Actual Outcome", "mean"),
).round(3)

# Sort in a logical order
order   = ["Low Risk", "Medium Risk", "High Risk"]
summary = summary.reindex([o for o in order if o in summary.index])

print("- Risk distribution across tst applicants -")
print(summary.to_string())

- Risk distribution across tst applicants -
               Count  Avg_Risk_Score  Actual_Approval
Risk Category                                        
Low Risk          67           7.804            0.910
Medium Risk       18          43.844            0.778
High Risk         38          87.647            0.263


In [48]:
# Scores one individual applicants
# This is what the system does in practice for a single person.
# Pass in one row of applicant data and get back a full risk verdict: probability, score, category, and recommendation.

def score_applicant(model, applicant_row):
    prob         = model.predict_proba(applicant_row)[0][1]
    risk_score   = round((1 - prob) * 100, 1)
    category     = get_risk_category(risk_score)
    recommendation = get_recommendation(risk_score)

    print("─" * 40)
    print(f"  Approval Probability : {prob:.1%}")
    print(f"  Risk Score           : {risk_score} / 100")
    print(f"  Risk Category        : {category}")
    print(f"  Recommendation       : {recommendation}")
    print("─" * 40)


# Tests it on applicant number 5 from the test set
print("- APPLICANT RISK ASSESSMENT -")
score_applicant(best_model, X_test.iloc[[5]])

- APPLICANT RISK ASSESSMENT -
────────────────────────────────────────
  Approval Probability : 52.9%
  Risk Score           : 47.1 / 100
  Risk Category        : Medium Risk
  Recommendation       : Review — officer should verify manually
────────────────────────────────────────


In [49]:
FEATURE_LABELS = {
    "gender":               "Gender",
    "married":              "Married",
    "dependents":           "Dependents",
    "education":            "Education",
    "self_employed":        "Self-Employed",
    "applicant_income":     "Applicant Income",
    "co_applicant_income":  "Co-applicant Income",
    "loan_amount":          "Loan Amount",
    "loan_amount_term":     "Loan Term",
    "credit_history":       "Credit History",
    "property_area":        "Property Area",
    "total_income":         "Total Income",
    "income_loan_ratio":    "Income-to-Loan Ratio",
    "total_income_log":     "Log Total Income",
    "loan_amount_log":      "Log Loan Amount",
}

# Get importances from the best model
importances = best_model.feature_importances_

df_importance = pd.DataFrame({
    "Feature":    [FEATURE_LABELS[f] for f in FEATURES],
    "Importance": importances
}).sort_values("Importance", ascending=False)

print("\n=== FEATURE IMPORTANCE — " + best_name + " ===")
print(df_importance.to_string(index=False))


=== FEATURE IMPORTANCE — LightGBM ===
             Feature  Importance
    Applicant Income         542
Income-to-Loan Ratio         512
        Total Income         407
         Loan Amount         315
 Co-applicant Income         254
     Log Loan Amount         144
          Dependents         109
      Credit History         105
           Loan Term         103
       Property Area         103
             Married          44
           Education          27
       Self-Employed          24
    Log Total Income          24
              Gender          20


In [50]:
# SHAP values — Global View
# SHAP goes deeper than feature importance.
# It calculates for every single prediction: "How much did each feature push the probability up or down?"

# The global view averages this across all test rows. It shows which features matter most and in which direction.

# Positive SHAP = pushed toward approval
# Negative SHAP = pushed toward rejection

print("\nComputing SHAP values...")

explainer = shap.TreeExplainer(best_model)
shap_vals  = explainer.shap_values(X_test)

# For binary classification, we want the approval class (class 1)
if isinstance(shap_vals, list):
    sv = shap_vals[1]           # class 1 = approval
else:
    sv = shap_vals

# Average absolute SHAP value per feature = average impact
mean_shap = np.abs(sv).mean(axis=0)

df_shap = pd.DataFrame({
    "Feature":     [FEATURE_LABELS[f] for f in FEATURES],
    "Mean |SHAP|": mean_shap.round(4),
}).sort_values("Mean |SHAP|", ascending=False)

print(" GLOBAL SHAP IMPORTANCE ")
print(df_shap.to_string(index=False))


Computing SHAP values...
 GLOBAL SHAP IMPORTANCE 
             Feature  Mean |SHAP|
      Credit History       2.7218
    Applicant Income       0.8054
         Loan Amount       0.6803
Income-to-Loan Ratio       0.6669
        Total Income       0.5202
     Log Loan Amount       0.4287
          Dependents       0.3657
       Property Area       0.3648
 Co-applicant Income       0.3532
             Married       0.2684
           Loan Term       0.1886
       Self-Employed       0.0837
    Log Total Income       0.0790
              Gender       0.0768
           Education       0.0383


In [51]:
# SHAP VALUES — One applicant
# For one specific applicant, SHAP shows exactly what pushed their decision toward approval or rejection, and by how much.


# Positive SHAP value → pushed toward APPROVAL (green)
# Negative SHAP value → pushed toward REJECTION (red)

def explain_applicant(model, explainer, applicant_row, applicant_index=0):
    shap_single = explainer.shap_values(applicant_row)

    if isinstance(shap_single, list):
        sv = shap_single[1][0]
    else:
        sv = shap_single[0]

    prob       = model.predict_proba(applicant_row)[0][1]
    risk_score = round((1 - prob) * 100, 1)
    decision   = "APPROVED" if prob >= 0.5 else "REJECTED"

    df_explain = pd.DataFrame({
        "Feature":    [FEATURE_LABELS[f] for f in FEATURES],
        "Value":      applicant_row.values[0],
        "SHAP Value": sv.round(4),
        "Direction":  ["↑ Toward Approval" if v > 0 else "↓ Toward Rejection" for v in sv],
    }).sort_values("SHAP Value", key=abs, ascending=False)

    print(f" APPLICANT {applicant_index} — DECISION EXPLANATION ")
    print(f"  Decision             : {decision}")
    print(f"  Approval Probability : {prob:.1%}")
    print(f"  Risk Score           : {risk_score} / 100")
    print(f"  Risk Category        : {get_risk_category(risk_score)}")
    print(f"\n  Top factors that drove this decision:")
    print(df_explain.head(8).to_string(index=False))


# Explain the first applicant in the test set
explain_applicant(best_model, explainer, X_test.iloc[[0]], applicant_index=0)

 APPLICANT 0 — DECISION EXPLANATION 
  Decision             : REJECTED
  Approval Probability : 0.0%
  Risk Score           : 100.0 / 100
  Risk Category        : High Risk

  Top factors that drove this decision:
             Feature       Value  SHAP Value          Direction
      Credit History    0.000000     -5.3588 ↓ Toward Rejection
    Applicant Income 6277.000000     -1.4075 ↓ Toward Rejection
Income-to-Loan Ratio   53.194915     -0.6535 ↓ Toward Rejection
       Property Area    0.000000     -0.4633 ↓ Toward Rejection
          Dependents    0.000000     -0.3878 ↓ Toward Rejection
         Loan Amount  118.000000      0.3687  ↑ Toward Approval
     Log Loan Amount    4.770685     -0.2572 ↓ Toward Rejection
        Total Income 6277.000000      0.2197  ↑ Toward Approval


In [52]:
print("\n── MODEL SAVED ──")
print("  best_model.joblib")


── MODEL SAVED ──
  best_model.joblib
